#  Preparação de Dados - Trabalho Estudantil
## Extração e Tratamento das Variáveis Q007 e Q008

**Notebook 2/7** - Série: Trabalho Estudantil e Desempenho no ENEM

---

##  Objetivos

1. Carregar o dataset original do ENEM 2023
2. Extrair variáveis relacionadas ao trabalho estudantil (Q007, Q008)
3. Integrar com variáveis já processadas (notas, socioeconômicas)
4. Realizar limpeza e tratamento de dados
5. Criar variáveis derivadas para análise
6. Salvar dataset processado para análises posteriores

---

## 1⃣ Setup Inicial

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

print(" Bibliotecas carregadas")

In [ ]:
# Definir caminhos
PROJECT_ROOT = Path('/home/interas/faculdade/ciencia-dados/enem-data-exploration')
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'interim' / 'unzipped_2023' / 'DADOS'
PROCESSED_DIR = DATA_DIR / 'processed'

# Arquivos
RAW_FILE = RAW_DIR / 'MICRODADOS_ENEM_2023.csv'
EXISTING_FILE = PROCESSED_DIR / 'enem_2023.parquet'  # Dataset já processado
OUTPUT_FILE = PROCESSED_DIR / 'enem_2023_trabalho_estudantil.parquet'

print(f" Dataset original: {RAW_FILE.exists()}")
print(f" Dataset processado existente: {EXISTING_FILE.exists()}")

---

## 2⃣ Carregar Dataset Existente

Primeiro, vamos carregar o dataset já processado que contém as notas e variáveis socioeconômicas básicas:

In [ ]:
%%time
# Carregar dataset já processado
df_base = pd.read_parquet(EXISTING_FILE)

print(f"\n Dataset base carregado:")
print(f"  Linhas: {len(df_base):,}")
print(f"  Colunas: {len(df_base.columns)}")
print(f"\n Colunas disponíveis:")
print(df_base.columns.tolist())

In [ ]:
# Primeiras linhas
df_base.head()

In [ ]:
# Informações sobre o dataset
df_base.info()

---

## 3⃣ Extrair Variáveis de Trabalho do Dataset Original

Agora vamos ler apenas as colunas Q007 e Q008 do dataset original:

In [ ]:
%%time
# Colunas a serem extraídas
colunas_trabalho = ['NU_INSCRICAO', 'Q007', 'Q008']

# Ler apenas as colunas necessárias
df_trabalho = pd.read_csv(
    RAW_FILE,
    sep=';',
    encoding='latin1',
    usecols=colunas_trabalho,
    low_memory=False
)

print(f"\n Variáveis de trabalho extraídas:")
print(f"  Linhas: {len(df_trabalho):,}")
print(f"  Colunas: {list(df_trabalho.columns)}")

In [ ]:
# Primeiras linhas
df_trabalho.head(10)

In [ ]:
# Verificar valores únicos
print(" Q007 - Situação de trabalho:")
print(df_trabalho['Q007'].value_counts(dropna=False).sort_index())

print("\n Q008 - Carga horária:")
print(df_trabalho['Q008'].value_counts(dropna=False).sort_index())

---

## 4⃣ Integrar Dados

Vamos juntar as variáveis de trabalho com o dataset base:

In [ ]:
# Verificar se NU_INSCRICAO existe no dataset base
if 'NU_INSCRICAO' in df_base.columns:
    print(" NU_INSCRICAO encontrado no dataset base")
    coluna_join = 'NU_INSCRICAO'
else:
    # Se não existir, precisamos ler do arquivo original
    print(" NU_INSCRICAO não encontrado. Lendo índice do arquivo original...")
    df_inscricao = pd.read_csv(
        RAW_FILE,
        sep=';',
        encoding='latin1',
        usecols=['NU_INSCRICAO'],
        low_memory=False
    )
    df_base['NU_INSCRICAO'] = df_inscricao['NU_INSCRICAO']
    coluna_join = 'NU_INSCRICAO'
    print(" NU_INSCRICAO adicionado ao dataset base")

In [ ]:
# Realizar merge
df = df_base.merge(df_trabalho, on=coluna_join, how='left')

print(f"\n Merge concluído:")
print(f"  Linhas: {len(df):,}")
print(f"  Colunas: {len(df.columns)}")
print(f"\n Novas colunas adicionadas: Q007, Q008")

In [ ]:
# Verificar dados ausentes
print(" Valores ausentes:")
print(df[['Q007', 'Q008']].isnull().sum())
print(f"\nTaxa de preenchimento Q007: {(1 - df['Q007'].isnull().mean()) * 100:.2f}%")
print(f"Taxa de preenchimento Q008: {(1 - df['Q008'].isnull().mean()) * 100:.2f}%")

---

## 5⃣ Tratamento de Dados

### 5.1 Análise de Valores Ausentes

In [ ]:
# Verificar se valores ausentes são aleatórios ou sistemáticos
print(" Análise de valores ausentes em Q007:")
print(f"\nTotal de ausentes: {df['Q007'].isnull().sum():,} ({df['Q007'].isnull().mean()*100:.2f}%)")

# Comparar com outras variáveis
print("\n Correlação de ausência com outras variáveis:")
if 'Q006' in df.columns:  # Renda
    print(f"  Q006 (Renda) também ausente: {(df['Q007'].isnull() & df['Q006'].isnull()).sum():,}")
if 'TP_ESCOLA' in df.columns:
    print(f"  Distribuição por escola:")
    print(df.groupby('TP_ESCOLA')['Q007'].apply(lambda x: x.isnull().mean() * 100))

### 5.2 Criar Labels Descritivos

In [ ]:
# Mapeamento Q007 - Situação de trabalho
Q007_LABELS = {
    'A': 'Não trabalho',
    'B': 'Trabalho eventualmente',
    'C': 'Trabalho meio período',
    'D': 'Trabalho período integral'
}

# Mapeamento Q008 - Carga horária
Q008_LABELS = {
    'A': 'Nenhuma',
    'B': 'Até 10 horas',
    'C': '11 a 20 horas',
    'D': '21 a 30 horas',
    'E': '31 a 40 horas',
    'F': 'Mais de 40 horas'
}

# Aplicar mapeamentos
df['Q007_label'] = df['Q007'].map(Q007_LABELS)
df['Q008_label'] = df['Q008'].map(Q008_LABELS)

print(" Labels descritivos criados")
print("\n Q007_label:")
print(df['Q007_label'].value_counts(dropna=False))
print("\n Q008_label:")
print(df['Q008_label'].value_counts(dropna=False))

### 5.3 Criar Variáveis Ordinais

In [ ]:
# Codificação ordinal Q007 (0 = não trabalha, 3 = período integral)
Q007_ORDINAL = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
df['Q007_ord'] = df['Q007'].map(Q007_ORDINAL)

# Codificação ordinal Q008 (0 = nenhuma, 5 = mais de 40h)
Q008_ORDINAL = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5}
df['Q008_ord'] = df['Q008'].map(Q008_ORDINAL)

print(" Variáveis ordinais criadas")
print("\n Distribuição Q007_ord:")
print(df['Q007_ord'].value_counts(dropna=False).sort_index())
print("\n Distribuição Q008_ord:")
print(df['Q008_ord'].value_counts(dropna=False).sort_index())

### 5.4 Criar Variáveis Derivadas

In [ ]:
# Variável binária: trabalha ou não
df['TRABALHA'] = (df['Q007'] != 'A').astype(int)

# Categorias simplificadas de trabalho
def categorizar_trabalho(row):
    if pd.isna(row['Q007']):
        return np.nan
    elif row['Q007'] == 'A':
        return 'Não trabalha'
    elif row['Q007'] in ['B', 'C']:
        return 'Trabalho parcial'
    else:  # D
        return 'Período integral'

df['CATEGORIA_TRABALHO'] = df.apply(categorizar_trabalho, axis=1)

# Carga horária numérica (ponto médio dos intervalos)
CARGA_HORARIA_NUM = {
    'A': 0,
    'B': 5,    # Até 10h -> média 5h
    'C': 15.5, # 11-20h -> média 15.5h
    'D': 25.5, # 21-30h -> média 25.5h
    'E': 35.5, # 31-40h -> média 35.5h
    'F': 45    # Mais de 40h -> estimativa 45h
}
df['CARGA_HORARIA_NUM'] = df['Q008'].map(CARGA_HORARIA_NUM)

print(" Variáveis derivadas criadas")
print("\n TRABALHA:")
print(df['TRABALHA'].value_counts())
print("\n CATEGORIA_TRABALHO:")
print(df['CATEGORIA_TRABALHO'].value_counts())
print("\n CARGA_HORARIA_NUM - Estatísticas:")
print(df['CARGA_HORARIA_NUM'].describe())

---

## 6⃣ Validação de Consistência

Verificar se há inconsistências entre Q007 e Q008:

In [ ]:
# Criar tabela cruzada
print(" Tabela Cruzada Q007 × Q008:")
tabela_cruzada = pd.crosstab(
    df['Q007_label'], 
    df['Q008_label'], 
    margins=True,
    dropna=False
)
print(tabela_cruzada)

# Identificar inconsistências
# Exemplo: Q007='A' (não trabalha) mas Q008 != 'A' (tem carga horária)
inconsistencias = df[(df['Q007'] == 'A') & (df['Q008'] != 'A') & df['Q008'].notna()]
print(f"\n Inconsistências encontradas: {len(inconsistencias):,}")
print(f"   ({len(inconsistencias)/len(df)*100:.2f}% do total)")

if len(inconsistencias) > 0:
    print("\nExemplos de inconsistências:")
    print(inconsistencias[['Q007', 'Q007_label', 'Q008', 'Q008_label']].head(10))

In [ ]:
# Tratar inconsistências (opcional)
# Se pessoa diz não trabalhar (Q007=A) mas tem carga horária, corrigir
print(" Corrigindo inconsistências...")

# Criar cópia das variáveis originais
df['Q007_original'] = df['Q007']
df['Q008_original'] = df['Q008']

# Regra: Se Q007=A mas Q008 tem valor diferente de A, 
# assumir que Q008 está correto e ajustar Q007
mask_inconsistente = (df['Q007'] == 'A') & (df['Q008'].notna()) & (df['Q008'] != 'A')
if mask_inconsistente.sum() > 0:
    # Mapear Q008 para Q007 aproximado
    def q008_para_q007(q008):
        if q008 in ['B', 'C']:  # Até 20h
            return 'C'  # Meio período
        elif q008 in ['D', 'E', 'F']:  # Mais de 20h
            return 'D'  # Período integral
        return 'A'
    
    df.loc[mask_inconsistente, 'Q007'] = df.loc[mask_inconsistente, 'Q008'].apply(q008_para_q007)
    print(f"   Corrigidos: {mask_inconsistente.sum():,} registros")
    
    # Atualizar variáveis derivadas
    df.loc[mask_inconsistente, 'Q007_label'] = df.loc[mask_inconsistente, 'Q007'].map(Q007_LABELS)
    df.loc[mask_inconsistente, 'Q007_ord'] = df.loc[mask_inconsistente, 'Q007'].map(Q007_ORDINAL)
    df.loc[mask_inconsistente, 'TRABALHA'] = 1
    df.loc[mask_inconsistente, 'CATEGORIA_TRABALHO'] = df.loc[mask_inconsistente].apply(categorizar_trabalho, axis=1)

print("\n Inconsistências tratadas")

---

## 7⃣ Estatísticas Finais

In [ ]:
print(" ESTATÍSTICAS FINAIS DO DATASET")
print("=" * 60)

print(f"\n Tamanho do dataset:")
print(f"  Total de registros: {len(df):,}")
print(f"  Total de colunas: {len(df.columns)}")

print(f"\n Situação de trabalho (Q007):")
print(df['Q007_label'].value_counts())
print(f"\n  Percentuais:")
print(df['Q007_label'].value_counts(normalize=True) * 100)

print(f"\n Carga horária (Q008):")
print(df['Q008_label'].value_counts())
print(f"\n  Percentuais:")
print(df['Q008_label'].value_counts(normalize=True) * 100)

print(f"\n Trabalha ou não:")
print(df['TRABALHA'].value_counts())
print(f"\n  Percentual que trabalha: {df['TRABALHA'].mean() * 100:.2f}%")

print(f"\n Categoria de trabalho:")
print(df['CATEGORIA_TRABALHO'].value_counts())

In [ ]:
# Resumo das novas colunas criadas
print("\n COLUNAS CRIADAS NESTE NOTEBOOK:")
print("=" * 60)
novas_colunas = [
    'Q007', 'Q008',  # Originais
    'Q007_label', 'Q008_label',  # Labels
    'Q007_ord', 'Q008_ord',  # Ordinais
    'TRABALHA',  # Binária
    'CATEGORIA_TRABALHO',  # Categórica simplificada
    'CARGA_HORARIA_NUM',  # Numérica
    'Q007_original', 'Q008_original'  # Backup
]
for col in novas_colunas:
    if col in df.columns:
        print(f"   {col}")

---

## 8⃣ Salvar Dataset Processado

In [ ]:
# Garantir que o diretório existe
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Salvar em formato parquet (mais eficiente)
print(f" Salvando dataset processado...")
df.to_parquet(OUTPUT_FILE, index=False, compression='snappy')

# Verificar tamanho do arquivo
tamanho_mb = OUTPUT_FILE.stat().st_size / (1024 * 1024)
print(f"\n Dataset salvo com sucesso!")
print(f"   Arquivo: {OUTPUT_FILE}")
print(f"   Tamanho: {tamanho_mb:.2f} MB")
print(f"   Registros: {len(df):,}")
print(f"   Colunas: {len(df.columns)}")

In [ ]:
# Também salvar uma versão CSV para facilitar inspeção
CSV_FILE = PROCESSED_DIR / 'enem_2023_trabalho_estudantil_sample.csv'

# Salvar amostra de 10.000 registros
df_sample = df.sample(n=10000, random_state=42)
df_sample.to_csv(CSV_FILE, index=False)

print(f"\n Amostra CSV salva: {CSV_FILE}")
print(f"   Registros: {len(df_sample):,}")

---

## 9⃣ Sumário de Qualidade dos Dados

In [ ]:
# Criar relatório de qualidade
print(" RELATÓRIO DE QUALIDADE DOS DADOS")
print("=" * 60)

colunas_trabalho = ['Q007', 'Q008', 'Q007_ord', 'Q008_ord', 'TRABALHA', 
                    'CATEGORIA_TRABALHO', 'CARGA_HORARIA_NUM']

for col in colunas_trabalho:
    if col in df.columns:
        total = len(df)
        nulos = df[col].isnull().sum()
        validos = total - nulos
        taxa_preenchimento = (validos / total) * 100
        
        print(f"\n{col}:")
        print(f"  Total: {total:,}")
        print(f"  Válidos: {validos:,} ({taxa_preenchimento:.2f}%)")
        print(f"  Nulos: {nulos:,} ({(nulos/total)*100:.2f}%)")
        
        if df[col].dtype in ['int64', 'float64']:
            print(f"  Média: {df[col].mean():.2f}")
            print(f"  Mediana: {df[col].median():.2f}")

---

##  Checklist de Conclusão

- [x] Dataset original carregado
- [x] Variáveis Q007 e Q008 extraídas
- [x] Merge realizado com sucesso
- [x] Labels descritivos criados
- [x] Variáveis ordinais criadas
- [x] Variáveis derivadas criadas (TRABALHA, CATEGORIA_TRABALHO, CARGA_HORARIA_NUM)
- [x] Inconsistências identificadas e tratadas
- [x] Dataset salvo em formato Parquet
- [x] Amostra CSV gerada para inspeção
- [x] Relatório de qualidade gerado

---

##  Próximos Passos

 **`03_analise_descritiva_trabalho.ipynb`**

No próximo notebook, iremos:
1. Analisar a distribuição de estudantes por situação de trabalho
2. Calcular estatísticas descritivas de desempenho por grupo
3. Criar visualizações comparativas
4. Traçar o perfil socioeconômico dos estudantes que trabalham

---

*Notebook criado em: 10 de dezembro de 2025*  
*Dataset processado: enem_2023_trabalho_estudantil.parquet*